In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Загружаем данные
df = pd.read_csv('../data/train.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']

# Разделяем на train/test (для экспериментов)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [6]:
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [7]:
categorical_features = ['Sex', 'Embarked', 'Pclass']

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [9]:
X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train_transformed.shape)  # должно быть (количество строк, количество признаков после one-hot)
# Первые 5 строк
print(X_train_transformed[:5])

(712, 13)
[[-0.08113533  0.5138115  -0.46508428 -0.46618317  0.          1.
   0.          0.          1.          0.          0.          0.
   1.        ]
 [-0.08113533 -0.66256323 -0.46508428 -0.46618317  0.          1.
   0.          0.          1.          0.          0.          1.
   0.        ]
 [-0.08113533  3.95539858 -0.46508428 -0.46618317  0.          1.
   0.          0.          1.          0.          1.          0.
   0.        ]
 [-0.88782719 -0.46787435 -0.46508428  0.72778236  1.          0.
   0.          0.          1.          0.          0.          0.
   1.        ]
 [ 0.11093416 -0.11597681  0.47833454  0.72778236  1.          0.
   0.          0.          1.          0.          0.          1.
   0.        ]]


In [10]:
def add_family_features(df):
    """Добавляет колонки FamilySize и IsAlone"""
    df_copy = df.copy()
    df_copy['FamilySize'] = df_copy['SibSp'] + df_copy['Parch'] + 1
    df_copy['IsAlone'] = (df_copy['FamilySize'] == 1).astype(int)
    return df_copy

In [11]:
from sklearn.preprocessing import FunctionTransformer

# 1. Трансформер для добавления признаков
family_adder = FunctionTransformer(add_family_features, validate=False)

# 2. Определим новые колонки для числовых и категориальных
numeric_features_final = ['Age', 'Fare', 'FamilySize', 'IsAlone']
categorical_features_final = ['Sex', 'Embarked', 'Pclass']

# 3. Создаём трансформеры как раньше
numeric_transformer_final = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer_final = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 4. ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_final, numeric_features_final),
        ('cat', categorical_transformer_final, categorical_features_final)
    ]
)

# 5. Полный предобработчик (сначала добавляем признаки, потом трансформируем)
full_preprocessor = Pipeline(steps=[
    ('add_family', family_adder),
    ('column_transform', column_transformer)
])

In [12]:
X_train_prep = full_preprocessor.fit_transform(X_train)
print(X_train_prep.shape)

(712, 13)


In [13]:
pipeline = Pipeline(steps=[
    ('preprocessor', full_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

# Оценка на тесте
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'ROC-AUC на тесте: {roc_auc:.4f}')

ROC-AUC на тесте: 0.8468


In [6]:
# Код с учётом импортов
import sys
import os

# Добавляем корневую папку проекта в путь, чтобы импортировать src
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

# Импортируем нашу функцию создания препроцессора из src
from src import create_preprocessor

print("Импорты выполнены успешно!")

Импорты выполнены успешно!


In [7]:
# Загружаем данные
df = pd.read_csv('../data/train.csv')

# Отделяем целевую переменную
X = df.drop('Survived', axis=1)
y = df['Survived']

print(f"Размер выборки: {X.shape}")
print(f"Баланс классов:\n{y.value_counts(normalize=True)}")

Размер выборки: (891, 11)
Баланс классов:
Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64


In [8]:
# Делим данные (стратификация сохраняет пропорцию выживших)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

Train size: (712, 11), Test size: (179, 11)


In [9]:
# Создаем экземпляр препроцессора
preprocessor = create_preprocessor()

# Применяем к обучающей выборке (fit_transform — вычисляем параметры и преобразуем)
X_train_prepared = preprocessor.fit_transform(X_train)

# К тестовой применяем только transform (не подглядываем в статистику теста)
X_test_prepared = preprocessor.transform(X_test)

print(f"Размер обучающей выборки после препроцессинга: {X_train_prepared.shape}")
print(f"Размер тестовой выборки после препроцессинга: {X_test_prepared.shape}")

# Посмотрим на первые 5 строк (это уже numpy-массив, а не DataFrame)
print("\nПример преобразованных данных (первые 5 строк):")
print(X_train_prepared[:5, :5])  # Покажем первые 5 колонок для краткости

Размер обучающей выборки после препроцессинга: (712, 13)
Размер тестовой выборки после препроцессинга: (179, 13)

Пример преобразованных данных (первые 5 строк):
[[-0.08113533  0.5138115  -0.55633858  0.80034555  0.        ]
 [-0.08113533 -0.66256323 -0.55633858  0.80034555  0.        ]
 [-0.08113533  3.95539858 -0.55633858  0.80034555  0.        ]
 [-0.88782719 -0.46787435  0.07341193 -1.24946032  1.        ]
 [ 0.11093416 -0.11597681  0.70316243 -1.24946032  1.        ]]


In [10]:
# Создаем модель
model = LogisticRegression(max_iter=1000, random_state=42)

# Обучаем на подготовленных данных
model.fit(X_train_prepared, y_train)

print("Модель обучена!")

Модель обучена!


In [11]:
# Предсказания вероятностей
y_pred_proba = model.predict_proba(X_test_prepared)[:, 1]
y_pred = model.predict(X_test_prepared)

# Метрики
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)

print(f"ROC-AUC на тесте: {roc_auc:.4f}")
print(f"Accuracy на тесте: {accuracy:.4f}")
print("\nДетальный отчет по классификации:")
print(classification_report(y_test, y_pred))

ROC-AUC на тесте: 0.8468
Accuracy на тесте: 0.8045

Детальный отчет по классификации:
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.78      0.68      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



In [12]:
# Получаем имена признаков после трансформации
# (OneHotEncoder создал названия вида Sex_male, Sex_female, Embarked_S и т.д.)
feature_names = preprocessor.named_steps['column_transform'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(
    input_features=['Sex', 'Embarked', 'Pclass']
)

# Добавляем числовые имена (Age, Fare, FamilySize, IsAlone)
num_features = ['Age', 'Fare', 'FamilySize', 'IsAlone']
all_features = np.concatenate([num_features, feature_names])

# Собираем в DataFrame
coef_df = pd.DataFrame({
    'feature': all_features,
    'coef': model.coef_[0]
}).sort_values('coef', ascending=False)

print("Топ-10 самых влиятельных признаков:")
print(coef_df.head(10))

Топ-10 самых влиятельных признаков:
             feature      coef
4         Sex_female  1.320549
10          Pclass_1  1.002300
7         Embarked_Q  0.280125
9   Embarked_missing  0.243510
1               Fare  0.114267
11          Pclass_2  0.102864
6         Embarked_C -0.034834
3            IsAlone -0.290148
8         Embarked_S -0.399179
0                Age -0.467734


In [ ]:
Анализ важности признаков (логистическая регрессия):

Самый сильный положительный вклад – Sex_female (коэф. 1.32) и Pclass_1 (1.00), что соответствует выводам EDA.

Отрицательно влияют возраст (Age, –0.47) и порт посадки S (Embarked_S, –0.40).

Признак IsAlone имеет отрицательный коэффициент (–0.29), подтверждая, что одиночные пассажиры выживали реже.

Модель интерпретируема и согласуется с логикой: женщины, дети и пассажиры первого класса имели преимущество.

In [13]:
# 1. Импорты
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix

# 2. Загрузка данных
df = pd.read_csv('../data/train.csv')

# 3. Отбор признаков
# Создаём признак "IsAlone" (одиночка ли пассажир)
df['IsAlone'] = (df['SibSp'] + df['Parch'] == 0).astype(int)

# Оставляем только нужные колонки
features = ['Age', 'Pclass', 'Sex', 'IsAlone']
X = df[features]
y = df['Survived']

print("Первые 5 строк отобранных признаков:")
print(X.head())

# 4. Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Предобработка (заполнение пропусков, кодирование категорий, масштабирование)
# Копируем, чтобы не менять исходный DataFrame
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

# 5a. Заполняем пропуски в Age медианой (обучаем на train, применяем к test)
age_median = X_train_processed['Age'].median()
X_train_processed['Age'] = X_train_processed['Age'].fillna(age_median)
X_test_processed['Age'] = X_test_processed['Age'].fillna(age_median)

# 5b. Кодируем Sex: male -> 0, female -> 1
X_train_processed['Sex'] = X_train_processed['Sex'].map({'male': 0, 'female': 1})
X_test_processed['Sex'] = X_test_processed['Sex'].map({'male': 0, 'female': 1})

# 5c. Масштабируем числовые признаки (Age, Pclass, IsAlone - хотя Pclass и так в порядке)
scaler = StandardScaler()
numeric_cols = ['Age', 'Pclass', 'IsAlone']
# Обучаем scaler на train, преобразуем train и test
X_train_processed[numeric_cols] = scaler.fit_transform(X_train_processed[numeric_cols])
X_test_processed[numeric_cols] = scaler.transform(X_test_processed[numeric_cols])

# 6. Обучение модели
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_processed, y_train)

# 7. Предсказание и оценка
y_pred = model.predict(X_test_processed)
y_pred_proba = model.predict_proba(X_test_processed)[:, 1]

print("\n=== Результаты на тестовой выборке ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 8. Коэффициенты модели (важность признаков)
coef_df = pd.DataFrame({
    'feature': features,
    'coef': model.coef_[0]
}).sort_values('coef', ascending=False)

print("\n=== Важность признаков (коэффициенты) ===")
print(coef_df)

Первые 5 строк отобранных признаков:
    Age  Pclass     Sex  IsAlone
0  22.0       3    male        0
1  38.0       1  female        0
2  26.0       3  female        1
3  35.0       1  female        0
4  35.0       3    male        1

=== Результаты на тестовой выборке ===
Accuracy: 0.7821
ROC-AUC:  0.8355

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.84      0.83       110
           1       0.73      0.70      0.71        69

    accuracy                           0.78       179
   macro avg       0.77      0.77      0.77       179
weighted avg       0.78      0.78      0.78       179


Confusion Matrix:
[[92 18]
 [21 48]]

=== Важность признаков (коэффициенты) ===
   feature      coef
2      Sex  2.530224
3  IsAlone  0.012673
0      Age -0.445512
1   Pclass -0.969262
